# PS 137 – Agricultural Production Scenario Coordinator
## Notebook 01 – Dataset Exploration (EDA)
**Member 1 Responsibility**

This notebook explores the raw agricultural dataset downloaded from Hugging Face.
We will understand the structure, quality, and patterns in the data.

> **Important:** We only load the raw Parquet file here. We do NOT modify it.

---
**Dataset:** India Public Agriculture  
**Source:** https://huggingface.co/datasets/therajasekhar/india-public-agriculture  
**File:** `data/raw/crop_statistics.parquet`

---
## Setup – Import Libraries

In [ ]:
# Standard libraries
import sys
from pathlib import Path

# Data manipulation
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_theme(style='whitegrid')

# ── Locate the project root (go up one level from notebooks/) ────────────────
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent  # PS137_Agricultural_Scenario_Coordinator/
PARQUET_PATH = PROJECT_ROOT / 'data' / 'raw' / 'crop_statistics.parquet'

print(f'Project root : {PROJECT_ROOT}')
print(f'Parquet file : {PARQUET_PATH}')
print(f'File exists  : {PARQUET_PATH.exists()}')

---
## Section 1 – Load the Dataset

We load the raw Parquet file. If it does not exist yet, run:
```
python src/data_collection/collect_dataset.py
```

In [ ]:
if not PARQUET_PATH.exists():
    raise FileNotFoundError(
        f'Raw Parquet file not found.\n'
        f'Expected: {PARQUET_PATH}\n'
        f'Run: python src/data_collection/collect_dataset.py'
    )

df = pd.read_parquet(PARQUET_PATH, engine='pyarrow')
print(f'Dataset loaded successfully.')
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

---
## Section 2 – Dataset Shape

**What this tells us:** How many records (rows) and how many features (columns) the dataset has.

In [ ]:
print('=' * 50)
print('DATASET SHAPE')
print('=' * 50)
print(f'Rows (records)  : {df.shape[0]:,}')
print(f'Columns         : {df.shape[1]}')
print()
print('Each row = one crop grown in one district in one year.')
print('Each column = one piece of information about that crop.')

---
## Section 3 – First Five Records

**What this tells us:** A quick preview of what the data looks like.

In [ ]:
print('First 5 rows of the dataset:')
df.head()

---
## Section 4 – Column Names and Data Types

**What this tells us:** The name of each column and what kind of data it stores
(text, number, integer, etc.).

In [ ]:
print('Column names and data types:')
print(df.dtypes)
print()
print('Explanation:')
print('  object  = text (string)')
print('  int64   = whole number (integer)')
print('  float64 = decimal number')

---
## Section 5 – Descriptive Statistics

**What this tells us:** Summary statistics for numeric columns –
mean (average), standard deviation (spread), minimum, maximum, and quartiles.

In [ ]:
print('Descriptive statistics for numeric columns:')
df.describe().round(2)

---
## Section 6 – Missing Values

**What this tells us:** Which columns have empty (NaN) values and how many.
Missing values need careful handling – we do NOT replace them blindly with zero.

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

if missing_df.empty:
    print('No missing values found in any column!')
else:
    print('Columns with missing values:')
    print(missing_df)
    print()
    # Visualise
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(missing_df.index, missing_df['Missing %'], color='coral')
    ax.set_xlabel('Column')
    ax.set_ylabel('Missing (%)')
    ax.set_title('Missing Value Percentage per Column')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()

---
## Section 7 – Duplicate Records

**What this tells us:** Whether any rows are exact copies of each other.
Duplicates can inflate statistics and need to be investigated before removal.

In [ ]:
n_exact_dupes = df.duplicated().sum()
print(f'Exact duplicate rows: {n_exact_dupes:,}')

key_cols = ['state', 'district', 'year', 'crop']
if all(c in df.columns for c in key_cols):
    n_key_dupes = df.duplicated(subset=key_cols).sum()
    print(f'Composite-key duplicates (state+district+year+crop): {n_key_dupes:,}')
    
    if n_key_dupes > 0:
        print()
        print('Sample of composite-key duplicate groups:')
        dupes = df[df.duplicated(subset=key_cols, keep=False)]
        print(dupes.head(10))

---
## Section 8 – Unique States

**What this tells us:** Which Indian states are covered in this dataset.

In [ ]:
if 'state' in df.columns:
    states = sorted(df['state'].dropna().unique())
    print(f'Number of unique states: {len(states)}')
    print()
    for s in states:
        print(f'  {s}')

---
## Section 9 – Unique Districts

**What this tells us:** How many districts are covered (a district is a subdivision of a state).

In [ ]:
if 'district' in df.columns:
    districts = sorted(df['district'].dropna().unique())
    print(f'Number of unique districts: {len(districts)}')
    print()
    print(f'First 20 districts (sorted alphabetically):')
    for d in districts[:20]:
        print(f'  {d}')
    if len(districts) > 20:
        print(f'  ... and {len(districts) - 20} more')

---
## Section 10 – Unique Crops

**What this tells us:** Which crops and crop categories are recorded in the dataset.

In [ ]:
if 'crop' in df.columns:
    crops = sorted(df['crop'].dropna().unique())
    print(f'Number of unique crops: {len(crops)}')
    print()
    for c in crops:
        print(f'  {c}')

---
## Section 11 – Year Coverage

**What this tells us:** Which years the data spans and how many records exist per year.

In [ ]:
if 'year' in df.columns:
    year_counts = df['year'].value_counts().sort_index()
    print(f'Year range: {df["year"].min()} – {df["year"].max()}')
    print()
    print('Records per year:')
    print(year_counts)
    print()
    # Bar chart
    fig, ax = plt.subplots()
    year_counts.plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title('Number of Records per Year')
    ax.set_xlabel('Year')
    ax.set_ylabel('Number of Records')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

---
## Section 12 – Zero-Value Analysis

**What this tells us:** How many records have zero area, production, or yield.
Zero production does NOT always mean an error – sometimes no crop was harvested.

In [ ]:
numeric_cols = ['area_1000_ha', 'production_1000_t', 'yield_kg_per_ha']

print('Zero-value counts:')
for col in numeric_cols:
    if col in df.columns:
        n_zero = (df[col] == 0).sum()
        pct = n_zero / len(df) * 100
        print(f'  {col:35s}: {n_zero:>7,} zeros  ({pct:.2f}%)')

# Records where BOTH area and production are zero
if 'area_1000_ha' in df.columns and 'production_1000_t' in df.columns:
    both_zero = ((df['area_1000_ha'] == 0) & (df['production_1000_t'] == 0)).sum()
    print(f'  Both area AND production = 0           : {both_zero:>7,} records')

---
## Section 13 – Negative-Value Analysis

**What this tells us:** Area, production, and yield cannot be negative in real life.
Any negative values indicate a data problem.

In [ ]:
print('Negative-value counts:')
found_negatives = False
for col in numeric_cols:
    if col in df.columns:
        n_neg = (df[col] < 0).sum()
        if n_neg > 0:
            found_negatives = True
            print(f'  [WARNING] {col}: {n_neg:,} negative values')
            print(df[df[col] < 0][[col, 'state', 'district', 'year', 'crop']].head(5))
        else:
            print(f'  [OK] {col}: no negative values')

if not found_negatives:
    print('  All numeric columns are free of negative values.')

---
## Section 14 – Min / Max Area

**What this tells us:** The smallest and largest area values in the dataset.

In [ ]:
if 'area_1000_ha' in df.columns:
    print('Area (thousand hectares):')
    print(f'  Minimum : {df["area_1000_ha"].min():.4f}  (1,000 ha)')
    print(f'  Maximum : {df["area_1000_ha"].max():.2f}  (1,000 ha)')
    print(f'  Mean    : {df["area_1000_ha"].mean():.2f}  (1,000 ha)')
    print()
    print('  Record with maximum area:')
    print(df.loc[df['area_1000_ha'].idxmax(), ['state', 'district', 'year', 'crop', 'area_1000_ha']])

---
## Section 15 – Min / Max Production

**What this tells us:** The smallest and largest production values.

In [ ]:
if 'production_1000_t' in df.columns:
    print('Production (thousand tonnes):')
    print(f'  Minimum : {df["production_1000_t"].min():.4f}')
    print(f'  Maximum : {df["production_1000_t"].max():.2f}')
    print(f'  Mean    : {df["production_1000_t"].mean():.2f}')
    print()
    print('  Record with maximum production:')
    print(df.loc[df['production_1000_t'].idxmax(), ['state', 'district', 'year', 'crop', 'production_1000_t']])

---
## Section 16 – Min / Max Yield

**What this tells us:** The range of yield values (kg per hectare).

In [ ]:
if 'yield_kg_per_ha' in df.columns:
    print('Yield (kg per hectare):')
    print(f'  Minimum : {df["yield_kg_per_ha"].min():.2f} kg/ha')
    print(f'  Maximum : {df["yield_kg_per_ha"].max():.2f} kg/ha')
    print(f'  Mean    : {df["yield_kg_per_ha"].mean():.2f} kg/ha')
    print()
    print('  Record with maximum yield:')
    print(df.loc[df['yield_kg_per_ha'].idxmax(), ['state', 'district', 'year', 'crop', 'yield_kg_per_ha']])
    print()
    # Histogram of yield
    fig, ax = plt.subplots()
    df['yield_kg_per_ha'].clip(upper=15000).hist(bins=50, ax=ax, color='forestgreen', edgecolor='white')
    ax.set_title('Distribution of Yield (kg/ha) – clipped at 15,000')
    ax.set_xlabel('Yield (kg/ha)')
    ax.set_ylabel('Number of Records')
    plt.tight_layout()
    plt.show()

---
## Section 17 – State-wise Production Analysis

**What this tells us:** Which states produce the most food (by total production across all years and crops).

In [ ]:
if 'state' in df.columns and 'production_1000_t' in df.columns:
    state_prod = (
        df.groupby('state')['production_1000_t']
        .sum()
        .sort_values(ascending=False)
        .reset_index()
    )
    state_prod.columns = ['State', 'Total Production (1000 t)']
    state_prod['Total Production (1000 t)'] = state_prod['Total Production (1000 t)'].round(1)
    
    print('Total production by state (all years, all crops):')
    print(state_prod.to_string(index=False))
    print()
    # Bar chart
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.bar(state_prod['State'], state_prod['Total Production (1000 t)'], color='steelblue')
    ax.set_title('Total Production by State (All Years, All Crops)')
    ax.set_xlabel('State')
    ax.set_ylabel('Total Production (Thousand Tonnes)')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

---
## Section 18 – Crop-wise Production Analysis

**What this tells us:** Which crops contribute the most to total production.

In [ ]:
if 'crop' in df.columns and 'production_1000_t' in df.columns:
    crop_prod = (
        df.groupby('crop')['production_1000_t']
        .sum()
        .sort_values(ascending=False)
        .reset_index()
    )
    crop_prod.columns = ['Crop', 'Total Production (1000 t)']
    crop_prod['Total Production (1000 t)'] = crop_prod['Total Production (1000 t)'].round(1)
    
    print('Top 15 crops by total production:')
    print(crop_prod.head(15).to_string(index=False))
    print()
    # Horizontal bar chart
    top15 = crop_prod.head(15)
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.barh(top15['Crop'][::-1], top15['Total Production (1000 t)'][::-1], color='darkorange')
    ax.set_title('Top 15 Crops by Total Production (All States, All Years)')
    ax.set_xlabel('Total Production (Thousand Tonnes)')
    plt.tight_layout()
    plt.show()

---
## Section 19 – Year-wise Production Analysis

**What this tells us:** How total agricultural production changed over the years covered by this dataset.

In [ ]:
if 'year' in df.columns and 'production_1000_t' in df.columns:
    year_prod = (
        df.groupby('year')['production_1000_t']
        .sum()
        .reset_index()
    )
    year_prod.columns = ['Year', 'Total Production (1000 t)']
    year_prod['Total Production (1000 t)'] = year_prod['Total Production (1000 t)'].round(1)
    
    print('Total production by year:')
    print(year_prod.to_string(index=False))
    print()
    # Line chart
    fig, ax = plt.subplots()
    ax.plot(year_prod['Year'], year_prod['Total Production (1000 t)'],
            marker='o', linewidth=2, color='seagreen')
    ax.set_title('Total Production by Year (All States, All Crops)')
    ax.set_xlabel('Year')
    ax.set_ylabel('Total Production (Thousand Tonnes)')
    ax.set_xticks(year_prod['Year'])
    plt.tight_layout()
    plt.show()

---
## Section 20 – EDA Summary

**Key findings from this notebook** (fill in after running all cells above):

In [ ]:
print('=' * 60)
print('EDA SUMMARY')
print('=' * 60)
print(f'Total records   : {len(df):,}')
print(f'Total columns   : {df.shape[1]}')
if 'state' in df.columns:
    print(f'Unique states   : {df["state"].nunique()}')
if 'district' in df.columns:
    print(f'Unique districts: {df["district"].nunique()}')
if 'crop' in df.columns:
    print(f'Unique crops    : {df["crop"].nunique()}')
if 'year' in df.columns:
    print(f'Year range      : {df["year"].min()} – {df["year"].max()}')

total_missing = df.isnull().sum().sum()
print(f'Total NaN cells : {total_missing:,}')
exact_dupes = df.duplicated().sum()
print(f'Exact duplicates: {exact_dupes:,}')
print('=' * 60)
print()
print('Next step: run preprocessing.')
print('  python src/preprocessing/preprocess_agriculture.py')